# Семинар 1a. Калибровка камеры

На лекции была модель камеры: точка мира переводится в систему камеры поворотом и сдвигом, потом матрица K переводит её в пиксели, а объектив добавляет дисторсию. Сегодня проверяем эту модель руками и учимся находить её параметры. Сначала синтетическая камера, где все параметры известны: проецируем, искажаем, калибруем и сравниваем с правдой. Смотрим, сколько нужно кадров, какие ракурсы полезны и что делает шум. Потом калибруем настоящую камеру по снимкам шахматной доски из примеров OpenCV, выпрямляем дисторсию и рисуем на доске трёхмерный куб.

Нужны `opencv-python`, `numpy`, `matplotlib`. Снимки доски (14 картинок по 30 КБ) скачиваются в `data/opencv` при первом запуске.

In [ ]:
import os
import urllib.request

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(suppress=True, precision=3)
print(np.__version__, cv.__version__)

## Проекция руками

Матрица K: фокусные расстояния в пикселях `fx`, `fy` и главная точка `cx`, `cy`. Камера смотрит вдоль оси Z. Точка (X, Y, Z) в системе камеры попадает в пиксель u = fx X / Z + cx, v = fy Y / Z + cy. Возьмём каркас куба со стороной 0.4 м в 2 метрах перед камерой.

In [ ]:
W, H = 640, 480
K = np.array([[800.0, 0, 320], [0, 800.0, 240], [0, 0, 1]])

cube = np.array([[x, y, z] for x in (-0.2, 0.2) for y in (-0.2, 0.2) for z in (0, 0.4)])
edges = [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(cube[i] != cube[j]) == 1]

def project(K, X):
    x = (K @ X.T).T
    return x[:, :2] / x[:, 2:]

def draw_cube(ax, uv, color="C0"):
    for i, j in edges:
        ax.plot(*uv[[i, j]].T, color=color)
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect("equal")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, Z, title in zip(axes, [2.0, 4.0, 1.0], ["Z = 2 м", "Z = 4 м", "Z = 1 м"]):
    uv = project(K, cube + [0.3, 0.1, Z])
    draw_cube(ax, uv)
    ax.set_title(title)
plt.show()
uv = project(K, cube + [0.3, 0.1, 2.0])
print("пиксели первых четырёх вершин:\n", uv[:4])

**Что видно:** чем дальше куб, тем он меньше: размер на картинке обратно пропорционален Z. В 2 метрах передняя грань куба (0.4 м) занимает 160 пикселей, а задняя, на 40 см дальше, 133 пикселя: перспектива. Вершины считаются по формуле руками: точка (0.1, −0.1, 2) даёт u = 800 · 0.1 / 2 + 320 = 360, v = 800 · (−0.1) / 2 + 240 = 200.

Две проверки на понимание. Если удвоить только глубину, картинка сожмётся к главной точке вдвое. Если умножить на 2 все координаты, то есть взять вдвое больший куб вдвое дальше, пиксели не изменятся совсем. По одной картинке абсолютный размер не узнать.

In [ ]:
P = cube + [0.3, 0.1, 2.0]
uv1 = project(K, P)
uv_deeper = project(K, P * [1, 1, 2])
uv_bigger = project(K, P * 2)
c = K[:2, 2]
print("удвоили Z: отношение расстояний до главной точки", np.round(np.linalg.norm(uv_deeper - c, axis=1) / np.linalg.norm(uv1 - c, axis=1), 3))
print("удвоили всё: максимальная разница пикселей", np.abs(uv_bigger - uv1).max())

## Поворот, сдвиг и `projectPoints`

Теперь куб задан в своей системе координат, а камера стоит где-то в мире. OpenCV описывает положение камеры двумя векторами: `rvec`, поворот в виде вектора Родрига (направление это ось, длина это угол в радианах), и `tvec`, сдвиг. Они переводят точку из системы объекта в систему камеры: X_cam = R X + t. Проверяем, что наша формула совпадает с `cv.projectPoints`.

In [ ]:
rvec = np.array([0.3, -0.4, 0.1])
tvec = np.array([0.1, 0.0, 2.0])
R, _ = cv.Rodrigues(rvec)
print("R =\n", R, "\nугол поворота, градусы:", np.rad2deg(np.linalg.norm(rvec)))
print("R ортогональна:", np.allclose(R @ R.T, np.eye(3)), " det =", round(np.linalg.det(R), 6))

ours = project(K, (R @ cube.T).T + tvec)
cv_uv, _ = cv.projectPoints(cube, rvec, tvec, K, None)
print("наша проекция против projectPoints, максимальная разница:", np.abs(ours - cv_uv[:, 0]).max())
fig, ax = plt.subplots(figsize=(6, 4.5))
draw_cube(ax, ours)
ax.set_title("куб, повёрнутый на 29°")
plt.show()

**Что видно:** вектор Родрига длиной 0.51 радиана это поворот на 29 градусов вокруг оси, заданной его направлением. Матрица поворота ортогональна и имеет определитель 1. Наша формула K (R X + t) совпадает с `cv.projectPoints` до ошибки округления 6e−14, то есть в OpenCV та же модель.

## Дисторсия

Реальный объектив гнёт прямые линии. Модель OpenCV: радиальная часть с коэффициентами k1, k2, k3 сдвигает точку вдоль луча от центра пропорционально r², r⁴, r⁶, тангенциальная часть p1, p2 описывает перекос линз. Посмотрим на сетку с k1 разного знака.

In [ ]:
gx, gy = np.meshgrid(np.linspace(-1.2, 1.2, 13), np.linspace(-0.9, 0.9, 10))
grid = np.stack([gx.ravel(), gy.ravel(), np.full(gx.size, 2.5)], 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, k1, title in zip(axes, [-0.3, 0.0, 0.3], ["k1 = −0.3: бочка", "k1 = 0", "k1 = 0.3: подушка"]):
    d = np.array([k1, 0, 0, 0, 0])
    p, _ = cv.projectPoints(grid, np.zeros(3), np.zeros(3), K, d)
    p = p[:, 0].reshape(10, 13, 2)
    for r in range(10):
        ax.plot(p[r, :, 0], p[r, :, 1], "C0", lw=0.8)
    for cidx in range(13):
        ax.plot(p[:, cidx, 0], p[:, cidx, 1], "C0", lw=0.8)
    ax.set_xlim(-50, W + 50); ax.set_ylim(H + 50, -50); ax.set_aspect("equal"); ax.set_title(title)
plt.show()
for k1 in [-0.3, 0.3]:
    p0, _ = cv.projectPoints(grid, np.zeros(3), np.zeros(3), K, None)
    p1, _ = cv.projectPoints(grid, np.zeros(3), np.zeros(3), K, np.array([k1, 0, 0, 0, 0]))
    sh = np.linalg.norm(p1 - p0, axis=2).ravel()
    r = np.linalg.norm(p0[:, 0] - K[:2, 2], axis=1)
    print(f"k1 = {k1:+.1f}: сдвиг в центре {sh[r.argmin()]:.2f} px, в углу {sh[r.argmax()]:.1f} px")

**Что видно:** при отрицательном k1 прямые выгибаются наружу, как бочка, при положительном внутрь, как подушка. В главной точке сдвига нет (0.02 пикселя), в углу кадра 52 пикселя: сдвиг растёт с расстоянием от центра как r³. Поэтому дисторсию не видно в центре кадра и очень видно по краям, особенно у широкоугольных объективов.

## Синтетическая калибровка

Доска 9 на 6 внутренних углов, клетка 25 мм. Координаты углов в системе доски известны: (i * 25 мм, j * 25 мм, 0). Снимаем доску синтетической камерой с известными K и дисторсией в 20 случайных положениях и добавляем к найденным углам шум 0.3 пикселя: примерно с такой точностью уточняются углы на настоящих снимках.

In [ ]:
PATTERN = (9, 6)
SQUARE = 0.025
board = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
board[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2) * SQUARE
K_true = np.array([[820.0, 0, 330], [0, 815.0, 245], [0, 0, 1]])
d_true = np.array([-0.25, 0.09, 0.001, -0.0005, 0.0])

def random_views(n, g, tilt=0.6, noise=0.3):
    objs, imgs, poses = [], [], []
    while len(imgs) < n:
        rvec = g.uniform(-tilt, tilt, 3) * [1, 1, 0.5]
        tvec = np.array([g.uniform(-0.12, 0.02), g.uniform(-0.08, 0.0), g.uniform(0.35, 0.7)])
        p, _ = cv.projectPoints(board, rvec, tvec, K_true, d_true)
        p = p[:, 0]
        if (p < 5).any() or (p[:, 0] > W - 5).any() or (p[:, 1] > H - 5).any():
            continue
        objs.append(board)
        imgs.append((p + g.normal(0, noise, p.shape)).astype(np.float32))
        poses.append((rvec, tvec))
    return objs, imgs, poses

objs, imgs, poses = random_views(20, np.random.default_rng(1))
fig, ax = plt.subplots(figsize=(7, 5.2))
for p in imgs:
    ax.plot(p[:, 0], p[:, 1], ".", ms=2)
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect("equal"); ax.set_title("углы доски на 20 синтетических кадрах")
plt.show()

In [ ]:
rms, K_est, d_est, rvecs, tvecs = cv.calibrateCamera(objs[:15], imgs[:15], (W, H), None, None)
print(f"RMS перепроекции на обучающих кадрах: {rms:.3f} px")
print("K найденная:\n", K_est)
print("K настоящая:\n", K_true)
print("дисторсия найденная:", d_est.ravel())
print("дисторсия настоящая:", d_true)

**Что видно:** по 15 кадрам с шумом 0.3 пикселя `calibrateCamera` нашёл fx = 818.5 и fy = 813.4 при правде 820 и 815, главную точку с ошибкой около 2 пикселей. RMS перепроекции 0.42 пикселя: это примерно шум углов, умноженный на √2, модель объясняет данные. Коэффициент k1 найден хорошо (−0.246 против −0.25), а k2 и k3 заметно ошиблись (0.13 и −0.006 вместо 0.09 и 0): они отвечают за края кадра, где углов доски мало, и частично заменяют друг друга.

## Проверка на новых кадрах

RMS на тех же кадрах, по которым калибровали, это ошибка подгонки, а не качество. Проверяем на 5 отложенных кадрах: K и дисторсию фиксируем, для каждого кадра находим только положение доски через `solvePnP` и меряем ошибку перепроекции. Ещё сравниваем, куда найденная модель переводит точки, с настоящей моделью по всему кадру.

In [ ]:
def reproj_rms(K_, d_, objs_, imgs_):
    errs = []
    for o, im in zip(objs_, imgs_):
        ok, rv, tv = cv.solvePnP(o, im, K_, d_)
        p, _ = cv.projectPoints(o, rv, tv, K_, d_)
        errs.append(np.sqrt(np.mean(np.sum((p[:, 0] - im) ** 2, 1))))
    return np.array(errs)

print("RMS на отложенных кадрах:", reproj_rms(K_est, d_est, objs[15:], imgs[15:]).round(3))

def undistort_error(K_, d_):
    us, vs = np.meshgrid(np.linspace(0, W, 33), np.linspace(0, H, 25))
    pix = np.stack([us.ravel(), vs.ravel()], 1).astype(np.float64)[:, None]
    a = cv.undistortPoints(pix, K_true, d_true, P=K_true)
    b = cv.undistortPoints(pix, K_, d_, P=K_true)
    return np.linalg.norm(a - b, axis=2).reshape(25, 33)

E = undistort_error(K_est, d_est)
fig, ax = plt.subplots(figsize=(6.5, 4.4))
im = ax.imshow(E, extent=[0, W, H, 0], cmap="magma")
plt.colorbar(im, ax=ax, label="px")
ax.set_title("разница с настоящей моделью после выпрямления")
plt.show()
print("в центре %.3f px, в среднем %.3f px, в худшем углу %.3f px" % (E[12, 16], E.mean(), E.max()))

**Что видно:** на 5 новых кадрах RMS 0.38 до 0.45, как на обучающих: калибровка не переобучилась. Но разница с настоящей моделью после выпрямления 3.4 пикселя даже в центре. Это не противоречие: в центре ошибка почти целиком из-за сдвига главной точки на 2 до 3 пикселя, а такой сдвиг доска компенсирует своим положением, и перепроекция его не замечает. В углах кадра ошибка 5.3 пикселя: там углов доски не было, и модель экстраполирует.

## Сколько кадров и каких

Повторяем калибровку на 3, 5, 10 и 15 кадрах, по 10 раз со случайным выбором кадров. Смотрим ошибку fx и ошибку модели в углах кадра.

In [ ]:
pool_o, pool_i, _ = random_views(60, np.random.default_rng(2))
for n in [3, 5, 10, 15]:
    fx_err, corner_err = [], []
    for rep in range(10):
        idx = np.random.default_rng(100 + rep).choice(60, n, replace=False)
        _, K_, d_, _, _ = cv.calibrateCamera([pool_o[i] for i in idx], [pool_i[i] for i in idx], (W, H), None, None)
        fx_err.append(abs(K_[0, 0] - K_true[0, 0]))
        corner_err.append(undistort_error(K_, d_).max())
    print(f"{n:2d} кадров: ошибка fx медиана {np.median(fx_err):6.2f} px, максимум {np.max(fx_err):6.2f};  ошибка в углу медиана {np.median(corner_err):5.2f} px")

**Что видно:** с 3 кадрами калибровка ненадёжна: медианная ошибка fx 6 пикселей, а в худшем из 10 повторов 67 пикселей, ошибка в углу кадра 30 пикселей. С 10 до 15 кадров медиана fx почти не меняется, а ошибка в углах продолжает падать с 10 до 6.5 пикселя. Главные точность и устойчивость дают первые 10 разных кадров, дальше важнее покрыть доской края кадра.

In [ ]:
front_o, front_i, _ = random_views(15, np.random.default_rng(3), tilt=0.02)
for name, (o, i) in {"фронтальные": (front_o, front_i), "с наклоном": (objs[:15], imgs[:15])}.items():
    rms_, K_, d_, _, _ = cv.calibrateCamera(o, i, (W, H), None, None)
    print(f"{name:12s} RMS {rms_:.3f} px, fx {K_[0, 0]:8.1f}, fy {K_[1, 1]:8.1f} (правда 820, 815)")

**Что видно:** 15 фронтальных кадров дают такой же маленький RMS 0.40, но fx = 889 вместо 820, ошибка 8%. Когда доска всегда параллельна сенсору, фокусное расстояние и расстояние до доски неразличимы: доска вдвое дальше при вдвое большем фокусе выглядит так же. Малый RMS ничего не говорит о правильности: нужны кадры с наклоном доски в разные стороны.

## Шум локализации углов

Те же 15 положений доски, разный шум углов.

In [ ]:
for noise in [0.05, 0.3, 1.0, 2.0]:
    o, i, _ = random_views(15, np.random.default_rng(1), noise=noise)
    rms_, K_, d_, _, _ = cv.calibrateCamera(o, i, (W, H), None, None)
    print(f"шум {noise:4.2f} px: RMS {rms_:.3f}, ошибка fx {abs(K_[0, 0] - 820):5.2f} px, k1 {d_[0, 0]:+.3f} (правда −0.250), ошибка модели в углу {undistort_error(K_, d_).max():.2f} px")

**Что видно:** RMS растёт ровно пропорционально шуму, около 1.4 шума. Ошибка fx тоже растёт линейно: 0.25 пикселя при шуме 0.05, почти 10 пикселей при шуме 2. Сильнее всего страдает дисторсия: при шуме 2 пикселя k1 = −0.14 вместо −0.25, и ошибка модели в углу кадра 53 пикселя. Поэтому углы уточняют `cornerSubPix`, а снимки делают резкими.

## Настоящая камера

Снимки шахматной доски из примеров OpenCV: 13 кадров 640 на 480, доска 9 на 6 внутренних углов. Находим углы, уточняем их до долей пикселя `cornerSubPix` и рисуем.

In [ ]:
os.makedirs("data/opencv", exist_ok=True)
names = [f"left{k:02d}.jpg" for k in range(1, 15) if k != 10]
URL = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/"
for n in names:
    if not os.path.exists("data/opencv/" + n):
        urllib.request.urlretrieve(URL + n, "data/opencv/" + n)
photos = [cv.imread("data/opencv/" + n, cv.IMREAD_GRAYSCALE) for n in names]
crit = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
real_o, real_i, used = [], [], []
for n, im in zip(names, photos):
    ok, c = cv.findChessboardCorners(im, PATTERN)
    if not ok:
        print("не найдено:", n)
        continue
    c2 = cv.cornerSubPix(im, c, (11, 11), (-1, -1), crit)
    real_o.append(board); real_i.append(c2.reshape(-1, 2)); used.append(n)
print(f"доска найдена на {len(used)} из {len(names)} кадров")
shift = np.concatenate([np.linalg.norm(a.reshape(-1, 2) - b, axis=1) for a, b in
                        zip([cv.findChessboardCorners(photos[names.index(n)], PATTERN)[1] for n in used], real_i)])
print("cornerSubPix сдвинул углы: медиана %.2f px, максимум %.2f px" % (np.median(shift), shift.max()))
fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
for ax, k in zip(axes, [0, 3, 6, 9]):
    vis = cv.cvtColor(photos[names.index(used[k])], cv.COLOR_GRAY2RGB)
    cv.drawChessboardCorners(vis, PATTERN, real_i[k][:, None].copy(), True)
    ax.imshow(vis); ax.set_title(used[k]); ax.axis("off")
plt.show()

**Что видно:** доска найдена на всех 13 кадрах. `cornerSubPix` сдвинул углы в среднем на 0.2 пикселя, но некоторые на 6.6 пикселя: грубый поиск углов на размытом кадре ошибается заметно. Цветные линии показывают порядок углов: он должен быть одинаковым на всех кадрах, иначе соответствие с точками доски перепутается. На left07 и left11 доска сильно повёрнута, и цветные строки идут в другую сторону. Порядок углов всё равно определён однозначно, потому что доска 9 на 6 углов не переходит в себя при повороте на 90°. С квадратной доской 7 на 7 так не получилось бы.

In [ ]:
rms_r, K_r, d_r, rv_r, tv_r = cv.calibrateCamera(real_o, real_i, (W, H), None, None)
print(f"RMS {rms_r:.3f} px")
print("K =\n", K_r)
print("дисторсия:", d_r.ravel())
fov = 2 * np.degrees(np.arctan(W / 2 / K_r[0, 0]))
print(f"горизонтальный угол обзора: {fov:.1f}°")
per_view = []
for o, im, rv, tv in zip(real_o, real_i, rv_r, tv_r):
    p, _ = cv.projectPoints(o, rv, tv, K_r, d_r)
    per_view.append(np.sqrt(np.mean(np.sum((p[:, 0] - im) ** 2, 1))))
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(used, per_view); ax.set_ylabel("RMS, px"); plt.xticks(rotation=45)
plt.show()

**Что видно:** RMS 0.41 пикселя, fx = fy = 536: пиксели квадратные. Главная точка (342, 236) немного сдвинута от центра кадра (320, 240), так бывает почти всегда. k1 = −0.27: заметная бочка. Горизонтальный угол обзора 62°, как у обычной веб-камеры. Один кадр, left02, выделяется: ошибка 1.2 пикселя против 0.2 у остальных. На нём доска сильно наклонена и частично смазана.

In [ ]:
loo = []
for k in range(len(real_o)):
    o = real_o[:k] + real_o[k + 1:]; i = real_i[:k] + real_i[k + 1:]
    _, K_, d_, _, _ = cv.calibrateCamera(o, i, (W, H), None, None)
    loo.append((K_[0, 0], reproj_rms(K_, d_, [real_o[k]], [real_i[k]])[0]))
loo = np.array(loo)
print("fx без одного кадра: от %.1f до %.1f" % (loo[:, 0].min(), loo[:, 0].max()))
print("RMS на выброшенном кадре: медиана %.3f px, максимум %.3f px" % (np.median(loo[:, 1]), loo[:, 1].max()))

**Что видно:** если выбросить любой один кадр, fx меняется в пределах 534 до 537: калибровка устойчива. Ошибка на выброшенном кадре в медиане 0.2 пикселя, как на обучающих, но у left02 1.24 пикселя. Такую проверку стоит делать всегда: кадр с большой ошибкой на leave-one-out надо посмотреть глазами и, скорее всего, выбросить.

## Выпрямление

`getOptimalNewCameraMatrix` подбирает новую матрицу для выпрямленного кадра: с `alpha=0` в кадре остаются только настоящие пиксели, с `alpha=1` сохраняются все исходные пиксели, и по краям появляются чёрные поля. Прямые края доски после выпрямления должны стать прямыми: проверим это, подгоняя прямую к каждой строке углов и меряя самое большое отклонение угла от неё.

In [ ]:
img = photos[names.index(used[0])]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("исходный")
for ax, alpha in zip(axes[1:], [0, 1]):
    newK, roi = cv.getOptimalNewCameraMatrix(K_r, d_r, (W, H), alpha)
    und = cv.undistort(img, K_r, d_r, None, newK)
    ax.imshow(und, cmap="gray"); ax.set_title(f"выпрямлен, alpha={alpha}")
for a in axes:
    a.axis("off")
plt.show()

def straightness(pts):
    rows = pts.reshape(PATTERN[1], PATTERN[0], 2)
    res = []
    for r in rows:
        c = r - r.mean(0)
        normal = np.linalg.svd(c)[2][-1]
        res.append(np.abs(c @ normal).max())
    return max(res)

before = [straightness(p) for p in real_i]
after = [straightness(cv.undistortPoints(p[:, None], K_r, d_r, P=K_r)[:, 0]) for p in real_i]
print("отклонение углов от прямой по строке доски: до %.2f px, после %.2f px (худший кадр)" % (max(before), max(after)))

**Что видно:** после выпрямления края доски и монитора становятся прямыми. С `alpha=0` картинка немного увеличена и обрезана, с `alpha=1` видны все исходные пиксели и чёрные изогнутые поля по краям. Числом: самый большой отрыв угла доски от прямой по строке был 3 пикселя, после выпрямления 0.4 пикселя, на уровне точности углов.

## Куб на доске

Калибровка нужна, чтобы по картинке находить положение предметов. Для каждого кадра `solvePnP` даёт положение доски относительно камеры, а по нему можно нарисовать на доске что угодно в трёх измерениях: например куб со стороной две клетки: основание зелёное, рёбра вверх голубые, верхняя грань красная.

In [ ]:
cube3 = np.array([[x, y, z] for x in (0, 2) for y in (0, 2) for z in (0, -2)], np.float32) * SQUARE + [3 * SQUARE, 2 * SQUARE, 0]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
for ax, k in zip(axes, [1, 5, 9]):
    ok, rv, tv = cv.solvePnP(real_o[k], real_i[k], K_r, d_r)
    p, _ = cv.projectPoints(cube3, rv, tv, K_r, d_r)
    p = p[:, 0]
    ax.imshow(photos[names.index(used[k])], cmap="gray")
    for i, j in [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(cube3[i] != cube3[j]) == 1]:
        col = "lime" if cube3[i, 2] == cube3[j, 2] == 0 else ("red" if cube3[i, 2] == cube3[j, 2] else "deepskyblue")
        ax.plot(*p[[i, j]].T, "-", color=col, lw=2)
    ax.set_title(used[k]); ax.axis("off")
plt.show()
for k in [1, 5, 9]:
    ok, rv, tv = cv.solvePnP(real_o[k], real_i[k], K_r, d_r)
    Rb, _ = cv.Rodrigues(rv)
    tilt = np.degrees(np.arccos(abs(Rb[2, 2])))
    print(f"{used[k]}: расстояние до угла доски {np.linalg.norm(tv) / SQUARE:.1f} клеток, доска наклонена к оси камеры на {tilt:.0f}°")

**Что видно:** куб стоит на доске на всех трёх кадрах, рёбра вверх перпендикулярны доске с учётом перспективы. Доска стоит примерно в 15 клетках от камеры, то есть если клетка 25 мм, в 37 см (размер клетки на этих снимках нам неизвестен, и метры без него не получить). Наклон доски к оси камеры от 26 до 41 градуса. Это та же задача, что в дополненной реальности: известная плоская метка, известная камера, `solvePnP`.

## Изменение размера и кроп

Калибровка привязана к разрешению. Если кадр уменьшить в 2 раза, fx, fy, cx, cy тоже делятся на 2. Если вырезать кусок, главная точка сдвигается. Дисторсия задана в нормированных координатах и не меняется. Проверим на настоящей модели.

In [ ]:
p_full, _ = cv.projectPoints(real_o[0], rv_r[0], tv_r[0], K_r, d_r)
K_half = K_r.copy(); K_half[:2] /= 2
p_half, _ = cv.projectPoints(real_o[0], rv_r[0], tv_r[0], K_half, d_r)
print("уменьшение в 2 раза: разница с половиной исходных пикселей", np.abs(p_half - p_full / 2).max().round(6))
x0, y0 = 100, 60
K_crop = K_r.copy(); K_crop[0, 2] -= x0; K_crop[1, 2] -= y0
p_crop, _ = cv.projectPoints(real_o[0], rv_r[0], tv_r[0], K_crop, d_r)
print("кроп со сдвигом (100, 60): разница", np.abs(p_crop - (p_full - [x0, y0])).max().round(6))

**Что видно:** при уменьшении картинки вдвое проекция с K, делённой на 2, точно равна половине исходных пикселей. При кропе достаточно сдвинуть главную точку. Дисторсию трогать не нужно. Частая ошибка в продукте: калибровали на 1920 на 1080, а сеть получает кадры 640 на 360 или кроп, и K забыли пересчитать.

*Попробуйте сами: откалибруйте камеру своего телефона. Распечатайте доску или покажите её на экране ноутбука, снимите 15 кадров с разных ракурсов и сравните найденный угол обзора с тем, что пишет производитель.*